## Variational Autoencoder Training and Evaluation

Small project demonstrating a basic deep learning workflow:
- train / test split
- VAE training
- evaluation on test data

Technologies: PyTorch, NumPy, Matplotlib

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split

1. Setup

In [2]:
BATCH_SIZE = 16
LR = 0.001
EPOCHS = 20
LATENT_DIM = 15

2. Data loading + Train / Test Split

In [3]:
transform = transforms.ToTensor()

train_data = datasets.FashionMNIST(
    root="./data", 
    train=True, 
    download=True, 
    transform=transform
)

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_data = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False
)

3. VAE definition 

In [4]:
class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        # Encoder
        self.fc1 = nn.Linear(28*28, 400)
        self.fc_mu = nn.Linear(400, latent_dim)
        self.fc_logvar = nn.Linear(400, latent_dim)

        # Decoder
        self.fc3 = nn.Linear(latent_dim, 400)
        self.fc4 = nn.Linear(400, 28*28)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.fc3(z))
        return torch.sigmoid(self.fc4(h))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        return self.decode(z), mu, logvar

4. Loss function

In [5]:
def vae_loss(recon_x, x, mu, logvar):
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction="sum")
    kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_div

5. Training loop

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = VAE().to(device)
optim = torch.optim.Adam(model.parameters(), lr= LR)

losses = []
for epoch in range(EPOCHS):
    model.train()
    for sample, _ in train_loader:
        sample = sample.view(-1, 28*28).to(device)
        
        reconstructed, mu, logvar = model(sample)
        loss = vae_loss(reconstructed, sample, mu, logvar)
        # recompute gradients:
        optim.zero_grad()  
        loss.backward() 
        optim.step() 
    losses.append(loss.item())

# Visualize Losses
plt.figure()
plt.plot(loss, marker="o")
plt.xlabel("Epoch")
plt.ylabel("VAE loss")
plt.title("Training Loss")
plt.show()

6. Evaluation

In [ ]:
model.eval()
with torch.no_grad():
    # Show 5 reconstructions of the test dataset
    for _ in range(5):
        sample, _ = next(iter(test_loader))
    
        sample = sample.to(device)
        sample_flat = sample.view(-1, 28*28)
    
        recon_x, _, _ = model(sample_flat)
    
        sample = sample.cpu()
        recon_x = recon_x.cpu().view(-1, 1, 28, 28)
    
        fig, axs = plt.subplots(2, 8, figsize=(12, 3))
    
        for i in range(8):
            axs[0, i].imshow(sample[i][0], cmap="gray")
            axs[0, i].axis("off")
    
            axs[1, i].imshow(recon_x[i][0], cmap="gray")
            axs[1, i].axis("off")
    
        axs[0, 0].set_ylabel("Original")
        axs[1, 0].set_ylabel("Reconstructed")
    
        plt.tight_layout()
        plt.show()

7. Generate new samples

In [ ]:
for _ in range(5):
        with torch.no_grad():
        z = torch.randn(8, 20).to(device)
        samples = model.decode(z).cpu().view(-1, 1, 28, 28)
    
    plt.figure(figsize=(8, 2))
    for i in range(8):
        plt.subplot(1, 8, i+1)
        plt.imshow(samples[i][0], cmap="gray")
        plt.axis("off")
    plt.show()

8. Conclusion

- This notebook deals with a variational autoencoder trained on the Fashion-MNIST dataset. The aim is for it to learn to independently learn a compact and continuous representation of grayscale images of clothing items.

- The small model has learned to compress the input images appropriately in order to achieve a low reconstruction loss. In addition, the model succeeds in enforcing a structured and coherent latent space with the help of Kullback-Leibler divergence regularization. 

- Due to the steady decrease in the loss function during training, stable optimization and convergence can be observed. This was confirmed by reconstructing unknown data, the test data set. The VAE has also learned to generalize well and thus perform well on data it has not yet seen. It achieves this by independently learning the overall structures and shapes of the garments. 

- The coherent latent space made it possible to generate a few realistic samples. However, some samples are blurred, which highlights a known limitation of standard VAEs when using pixel-wise reconstruction losses.

- Overall, the results confirm that the VAE is capable of learning meaningful latent representations for image data. Future work could focus on improving reconstruction quality through the use of convolutional architectures and increasing model capacity.